In [46]:
# PROJECT DAP PHASE 1 : Data preparation and cleaning 
# SECP3223
# SEC 01
# DR NOR ERNE NAZIRA BINTI BAZIN

# PRAVINRAJ A/L SIVABATHI (A23CS0171)
# AFIF SHAQIR IRFAN BIN ARQAM (A23CS0204)
# DHESHIEGAN SARAVANA MOORTHY (A23CS0072)



In [47]:
# 1. Import and load datasets

import pandas as pd

df_cars = pd.read_csv('cars_2025.csv')
df_fuel = pd.read_csv('FuelConsumption.csv')

print("Car Registrations Data Shape:", df_cars.shape)
print("Emission Data Shape:", df_fuel.shape)
print("\nSample from Car Registrations:")
print(df_cars.sample(3))
print("\nSample from Emission Data:")
print(df_fuel.sample(3))

Car Registrations Data Shape: (263578, 7)
Emission Data Shape: (1067, 13)

Sample from Car Registrations:
          date_reg                      type    maker model  colour    fuel  \
202165  2025-04-10                   motokar   Proton  Iriz   white  petrol   
155632  2025-03-21  motokar_pelbagai_utiliti  Perodua  Alza  silver  petrol   
247143  2025-04-29                   motokar   Proton  Saga  silver  petrol   

              state  
202165  Rakan Niaga  
155632  Rakan Niaga  
247143  Rakan Niaga  

Sample from Emission Data:
      MODELYEAR    MAKE              MODEL VEHICLECLASS  ENGINESIZE  \
1061       2014   VOLVO               XC60  SUV - SMALL         3.2   
567        2014  JAGUAR  XKR-S CONVERTIBLE  MINICOMPACT         5.0   
56         2014    AUDI                 S4      COMPACT         3.0   

      CYLINDERS TRANSMISSION FUELTYPE  FUELCONSUMPTION_CITY  \
1061          6          AS6        X                  13.0   
567           8          AS6        Z             

In [48]:
# 2. standardized  keys
df_cars['make_std'] = df_cars['maker'].str.upper().str.strip()
df_cars['model_std'] = df_cars['model'].str.upper().str.strip()
df_fuel['make_std'] = df_fuel['MAKE'].str.upper().str.strip()
df_fuel['model_std'] = df_fuel['MODEL'].str.upper().str.strip()


In [49]:
# 3. Merge car registrations with emissions 
merged = df_cars.merge(
    df_fuel,
    how='left',
    on=['make_std', 'model_std']
)

print("\nMerge successful! Merged data shape:", merged.shape)


Merge successful! Merged data shape: (277343, 22)


In [50]:
# 4. Handle missing CO₂ values
#   a) isolate unmatched for manual review
missing_co2 = merged[ merged['CO2EMISSIONS'].isna() ]

#   b) drop rows with missing CO₂ for clean analysis
clean_data = merged.dropna(subset=['CO2EMISSIONS']).copy()


In [51]:
# 5. Optional: check for and drop exact duplicate rows
clean_data.drop_duplicates(inplace=True)

In [52]:
# 6. Aggregations & summaries

# 6a. Counts summary
total_regs     = len(df_cars)
matched_regs   = len(clean_data)
unmatched_regs = len(missing_co2)

print("Total registrations:   ", total_regs)
print("Matched records:       ", matched_regs)
print("Unmatched records:     ", unmatched_regs)

# 6b. Average CO₂ emissions by fuel type
avg_by_fuel = (
    clean_data
    .groupby('fuel')['CO2EMISSIONS']
    .mean()
    .sort_values()
    .reset_index()
)
print("\nAverage CO₂ by fuel type:")
print(avg_by_fuel.to_string(index=False))

# 6c. Average CO₂ emissions by state
avg_by_state = (
    clean_data
    .groupby('state')['CO2EMISSIONS']
    .mean()
    .sort_values()
    .reset_index()
)
print("\nAverage CO₂ by state:")
print(avg_by_state.to_string(index=False))


Total registrations:    263578
Matched records:        9212
Unmatched records:      251480

Average CO₂ by fuel type:
         fuel  CO2EMISSIONS
       petrol    196.747879
hybrid_petrol    200.969723
       diesel    221.150000

Average CO₂ by state:
            state  CO2EMISSIONS
           Pahang    182.205882
            Sabah    182.220930
           Perlis    182.387755
         Kelantan    183.099099
            Kedah    185.548571
       Terengganu    186.218182
           Melaka    191.977011
  Negeri Sembilan    192.200000
          Sarawak    193.130719
      Rakan Niaga    195.616549
     Pulau Pinang    197.304659
         Selangor    197.992701
            Johor    199.213542
            Perak    199.695312
W.P. Kuala Lumpur    208.012796


In [65]:
# 7a. Create MultiIndex of (state, fuel) with average CO₂ emissions
mi = (
    clean_data
    .groupby(['state', 'fuel'])['CO2EMISSIONS']
    .mean()
)

# 7b. Example: Show average CO₂ emissions by fuel type for 'Pahang'
print("\nJohor CO₂ by fuel type:")
if 'Pahang' in mi.index.get_level_values('state'):
    print(mi.loc['Johor'])
else:
    print("No data available for Pahang.")

# 7c. Pivot to wide form: states as rows, fuel types as columns
emissions_table = mi.unstack()
print("\nPivot table (state × fuel) of average CO₂:")
print(emissions_table.round(2).head())



Johor CO₂ by fuel type:
fuel
hybrid_petrol    217.833333
petrol           197.520833
Name: CO2EMISSIONS, dtype: float64

Pivot table (state × fuel) of average CO₂:
fuel             diesel  hybrid_petrol  petrol
state                                         
Johor               NaN         217.83  197.52
Kedah               NaN         177.75  185.92
Kelantan            NaN         184.00  183.06
Melaka              NaN         223.00  189.26
Negeri Sembilan     NaN         198.58  190.98


In [67]:
# 8. Simple side-by-side summary (state & fuel CO₂ averages)

state_stats = avg_by_state.reset_index()
fuel_stats = avg_by_fuel.reset_index()

# Combine both summaries side by side
combined_stats = pd.concat([state_stats, fuel_stats], axis=1)

print("\n[8] Combined CO₂ Summary (State & Fuel Type):")
print(combined_stats.head().to_string(index=False))



[8] Combined CO₂ Summary (State & Fuel Type):
 index    state  CO2EMISSIONS  index          fuel  CO2EMISSIONS
     0   Pahang    182.205882    0.0        petrol    196.747879
     1    Sabah    182.220930    1.0 hybrid_petrol    200.969723
     2   Perlis    182.387755    2.0        diesel    221.150000
     3 Kelantan    183.099099    NaN           NaN           NaN
     4    Kedah    185.548571    NaN           NaN           NaN
